# 01. 확률분포와 Likelihood Modeling

로봇 센서는 항상 노이즈를 포함한다. 통계적 모델링의 첫 단계는 센서값을 하나의 숫자가 아니라 확률분포로 보는 것이다.

$$z = h(x) + \epsilon, \qquad \epsilon \sim \mathcal{N}(0,\sigma^2)$$

여기서는 range sensor 예제로 Gaussian noise, likelihood, maximum likelihood estimation을 연결한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False


## 1. Range Sensor Noise를 분포로 보기

같은 거리를 반복 측정해도 측정값은 흩어진다. 평균은 sensor bias, 분산은 uncertainty를 나타낸다.

In [ ]:
np.random.seed(501)
true_distance = 4.0
bias = 0.18
sigma = 0.32
z = true_distance + bias + np.random.randn(120) * sigma

mu_hat = z.mean()
sigma_hat = z.std(ddof=1)
xs = np.linspace(2.8, 5.4, 300)
pdf = 1 / (sigma_hat * np.sqrt(2*np.pi)) * np.exp(-0.5 * ((xs - mu_hat) / sigma_hat) ** 2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(z, bins=18, density=True, alpha=0.65, color='tab:blue', label='measurements')
ax.plot(xs, pdf, color='black', lw=2, label='fitted Gaussian')
ax.axvline(true_distance, color='tab:green', lw=2, label='true distance')
ax.axvline(mu_hat, color='tab:red', lw=2, ls='--', label='sample mean')
ax.set_title('Range sensor measurements as a probability distribution')
ax.set_xlabel('measured distance [m]')
ax.set_ylabel('density')
ax.legend()
plt.tight_layout()
plt.savefig('assets/stat_01_range_sensor_distribution.png', dpi=160)
plt.show()

print('estimated mean:', round(mu_hat, 3))
print('estimated sigma:', round(sigma_hat, 3))

## 2. Likelihood로 파라미터 추정하기

관측 데이터 $z_{1:n}$이 주어졌을 때, 어떤 $\mu$가 가장 그럴듯한지 likelihood로 비교한다.

$$L(\mu)=p(z_{1:n}\mid \mu,\sigma)=\prod_i \mathcal{N}(z_i;\mu,\sigma^2)$$

계산 안정성을 위해 실제 구현에서는 log-likelihood를 사용한다.

In [ ]:
mu_grid = np.linspace(3.6, 4.7, 240)
known_sigma = sigma_hat
log_likelihood = []
for mu in mu_grid:
    ll = -0.5 * np.sum(((z - mu) / known_sigma) ** 2) - len(z) * np.log(known_sigma * np.sqrt(2*np.pi))
    log_likelihood.append(ll)
log_likelihood = np.array(log_likelihood)
mu_mle = mu_grid[np.argmax(log_likelihood)]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(mu_grid, log_likelihood, color='tab:purple', lw=2)
ax.axvline(mu_mle, color='tab:red', ls='--', label=f'MLE = {mu_mle:.2f}')
ax.axvline(true_distance, color='tab:green', label='true distance')
ax.set_title('Log-likelihood over possible sensor mean')
ax.set_xlabel('$\\mu$')
ax.set_ylabel('$\\log p(z_{1:n} \\mid \\mu)$')
ax.legend()
plt.tight_layout()
plt.savefig('assets/stat_01_log_likelihood.png', dpi=160)
plt.show()

## 3. 로보틱스 연결

| 개념 | 의미 | 로보틱스 활용 |
|---|---|---|
| Probability distribution | 측정값의 불확실성 표현 | sensor noise modeling |
| Likelihood | 특정 상태/파라미터에서 관측이 나올 확률 | localization, SLAM update |
| MLE | 데이터를 가장 잘 설명하는 파라미터 | sensor calibration |
| Log-likelihood | 곱셈을 덧셈으로 바꾼 안정적 계산 | optimization 기반 estimation |

Bayes filter의 $p(z_t\mid x_t)$도 결국 sensor likelihood model이다.